In [1]:
   %load_ext autoreload
   %autoreload 2

   import sqlite3
   import numpy as np
   import pandas as pd
   from model import load_games, load_market, fit_poisson, predict, backtest, add_scores

   conn = sqlite3.connect("data/sports.db")
   games = load_games(conn)
   market = load_market(conn)
   games.tail()

,id,season,date,home,away,home_score,away_score,home_xg,away_xg
12749,12747,2026,2026-09-19,Brighton,Arsenal,3,0,2.112680,1.23979
12750,12753,2026,2026-09-20,Man City,Sunderland,5,3,2.668560,3.10661
12751,12752,2026,2026-09-20,Leeds,Crystal Palace,0,0,1.085520,1.12905
12752,12754,2026,2026-09-20,Fulham,Man United,1,1,1.717190,2.12453
12753,12751,2026,2026-09-20,Bournemouth,Liverpool,0,1,0.944706,1.29116


In [2]:
as_of = games["date"].max()
train = games[games["date"] >= "2023-07-01"]
model = fit_poisson(train, as_of)

print(f"Base expected goals: {model['base']:.2f}")
print(f"Home advantage: x{model['home_adv']:.2f}")

current_teams = games.loc[games["season"] == 2026, "home"].unique()
model["ratings"].loc[current_teams].sort_values("attack", ascending=False).round(2)

Base expected goals: 1.29
Home advantage: x1.20


,attack,defense
Man City,1.52,1.32
Arsenal,1.36,1.72
Brighton,1.30,1.13
Man United,1.30,1.11
Chelsea,1.29,0.97
Liverpool,1.29,1.16
Brentford,1.23,1.08
Bournemouth,1.18,1.11
Newcastle,1.18,0.98
Aston Villa,1.09,1.02


In [3]:
p = predict(model, "Arsenal", "Man City")

print(f"Expected goals: Arsenal {p['exp_home']:.2f} - {p['exp_away']:.2f} Man City")
print(f"Home win {p['home']:.1%}   Draw {p['draw']:.1%}   Away win {p['away']:.1%}")
print(f"Fair odds: {1/p['home']:.2f} / {1/p['draw']:.2f} / {1/p['away']:.2f}")

pd.DataFrame(
    p["grid"][:5, :5] * 100,
    index=pd.Index(range(5), name="Arsenal goals"),
    columns=pd.Index(range(5), name="Man City goals"),
).round(1)

Expected goals: Arsenal 1.59 - 1.14 Man City
Home win 47.7%   Draw 24.9%   Away win 27.4%
Fair odds: 2.09 / 4.02 / 3.65


Man City goals,0,1,2,3,4
Arsenal goals,,,,,
0,6.5,7.4,4.2,1.6,0.5
1,10.3,11.8,6.7,2.6,0.7
2,8.2,9.4,5.4,2.0,0.6
3,4.4,5.0,2.8,1.1,0.3
4,1.7,2.0,1.1,0.4,0.1


In [4]:
market = pd.read_sql("""
    WITH closing AS (
        SELECT game_id, bookmaker,
               MAX(CASE WHEN outcome = 'home' THEN 1.0 / price END) AS h,
               MAX(CASE WHEN outcome = 'draw' THEN 1.0 / price END) AS d,
               MAX(CASE WHEN outcome = 'away' THEN 1.0 / price END) AS a
        FROM odds
        WHERE market = '1x2' AND is_closing = 1
          AND bookmaker IN ('Pinnacle', 'Betfair Exchange')
        GROUP BY game_id, bookmaker
        HAVING COUNT(*) = 3
    ),
    ranked AS (
        SELECT *, ROW_NUMBER() OVER (
                      PARTITION BY game_id
                      ORDER BY bookmaker = 'Pinnacle' DESC) AS rn
        FROM closing
    )
    SELECT game_id AS id, bookmaker AS benchmark,
           h / (h + d + a) AS mkt_home,
           d / (h + d + a) AS mkt_draw,
           a / (h + d + a) AS mkt_away
    FROM ranked
    WHERE rn = 1
""", conn)

market["benchmark"].value_counts()

benchmark
Pinnacle            5150
Betfair Exchange     198
Name: count, dtype: int64

In [5]:
tuning_games = games[games["season"] <= 2020]

rows = []
for half_life in [90, 180, 365, 730]:
    for shrinkage in [0.5, 1, 3, 10]:
        res = backtest(tuning_games, start_season=2016,
                       half_life_days=half_life, shrinkage=shrinkage)
        add_scores(res, "model")
        rows.append({"half_life": half_life, "shrinkage": shrinkage,
                     "logloss": res["model_logloss"].mean()})

tuning = pd.DataFrame(rows)
tuning.pivot(index="half_life", columns="shrinkage", values="logloss").round(4)

shrinkage,0.5,1.0,3.0,10.0
half_life,,,,
90,0.9554,0.9558,0.9612,0.9831
180,0.9517,0.9521,0.9553,0.9693
365,0.9530,0.9532,0.9550,0.9636
730,0.9554,0.9555,0.9566,0.9625


In [6]:
tuned = backtest(games, start_season=2021)
old = backtest(games, start_season=2021, half_life_days=180, shrinkage=3.0)

old = old[["id", "model_home", "model_draw", "model_away"]]
old.columns = ["id", "old_home", "old_draw", "old_away"]

test = tuned.merge(market, on="id").merge(old, on="id")
for prefix in ["model", "old", "mkt"]:
    add_scores(test, prefix)

print(f"Games: {len(test)}")
test[["old_logloss", "model_logloss", "mkt_logloss",
      "old_brier", "model_brier", "mkt_brier"]].mean().round(4)

Games: 1921


old_logloss      0.9781
model_logloss    0.9754
mkt_logloss      0.9592
old_brier        0.5815
model_brier      0.5800
mkt_brier        0.5697
dtype: float64

In [7]:
tuning_games = games[games["season"] <= 2020]

rows = []
for xg_weight in [0, 0.25, 0.5, 0.75, 1.0]:
    res = backtest(tuning_games, start_season=2016, xg_weight=xg_weight)
    add_scores(res, "model")
    rows.append({"xg_weight": xg_weight, "logloss": res["model_logloss"].mean()})

pd.DataFrame(rows).round(4)

,xg_weight,logloss
0,0.00,0.9584
1,0.25,0.9540
2,0.50,0.9517
3,0.75,0.9514
4,1.00,0.9533


In [8]:
xg_model = backtest(games, start_season=2021)
goals_model = backtest(games, start_season=2021, xg_weight=0)

goals_model = goals_model[["id", "model_home", "model_draw", "model_away"]]
goals_model.columns = ["id", "goals_home", "goals_draw", "goals_away"]

test = xg_model.merge(market, on="id").merge(goals_model, on="id")
for prefix in ["goals", "model", "mkt"]:
    add_scores(test, prefix)

scores = test[["goals_logloss", "model_logloss", "mkt_logloss"]].mean()
gap_closed = (scores["goals_logloss"] - scores["model_logloss"]) / (scores["goals_logloss"] - scores["mkt_logloss"])

print(f"Games: {len(test)}")
print(scores.round(4))
print(f"Share of the gap to the market closed by xG: {gap_closed:.0%}")

Games: 1921
goals_logloss    0.9826
model_logloss    0.9754
mkt_logloss      0.9592
dtype: float64
Share of the gap to the market closed by xG: 31%


In [9]:
tuning_games = games[games["season"] <= 2020]

rows = []
for rho in [0, -0.05, -0.1, -0.15, -0.2]:
    res = backtest(tuning_games, start_season=2016, rho=rho)
    add_scores(res, "model")
    rows.append({"rho": rho,
                 "logloss": res["model_logloss"].mean(),
                 "predicted_draws": res["model_draw"].mean(),
                 "actual_draws": (res["result"] == "draw").mean()})

pd.DataFrame(rows).round(4)

,rho,logloss,predicted_draws,actual_draws
0,0.00,0.9517,0.2347,0.2254
1,-0.05,0.9523,0.2456,0.2254
2,-0.10,0.9536,0.2564,0.2254
3,-0.15,0.9555,0.2673,0.2254
4,-0.20,0.9581,0.2782,0.2254


In [10]:
rows = []
for half_life in [120, 180, 270, 365, 540]:
    for shrinkage in [0.5, 1, 2]:
        res = backtest(tuning_games, start_season=2016,
                       half_life_days=half_life, shrinkage=shrinkage)
        add_scores(res, "model")
        rows.append({"half_life": half_life, "shrinkage": shrinkage,
                     "logloss": res["model_logloss"].mean()})

pd.DataFrame(rows).pivot(index="half_life", columns="shrinkage", values="logloss").round(4)

shrinkage,0.5,1.0,2.0
half_life,,,
120,0.9530,0.9535,0.9554
180,0.9517,0.9521,0.9535
270,0.9521,0.9524,0.9534
365,0.9530,0.9532,0.9540
540,0.9544,0.9545,0.9551


In [11]:
rows = []
for half_life in [180, 270]:
    for shrinkage in [0, 0.25, 0.5]:
        res = backtest(tuning_games, start_season=2016,
                       half_life_days=half_life, shrinkage=shrinkage)
        add_scores(res, "model")
        rows.append({"half_life": half_life, "shrinkage": shrinkage,
                     "logloss": res["model_logloss"].mean()})

pd.DataFrame(rows).pivot(index="half_life", columns="shrinkage", values="logloss").round(4)

shrinkage,0.00,0.25,0.50
half_life,,,
180,0.9519,0.9516,0.9517
270,0.9523,0.9520,0.9521


In [12]:
final = backtest(games, start_season=2021)
prev = backtest(games, start_season=2021, half_life_days=365, shrinkage=1.0)

prev = prev[["id", "model_home", "model_draw", "model_away"]]
prev.columns = ["id", "prev_home", "prev_draw", "prev_away"]

test = final.merge(market, on="id").merge(prev, on="id")
for prefix in ["prev", "model", "mkt"]:
    add_scores(test, prefix)

scores = test[["prev_logloss", "model_logloss", "mkt_logloss"]].mean()
goals_only = 0.9813  # tuned goals-only model on these same games (step 17)
gap_closed = (goals_only - scores["model_logloss"]) / (goals_only - scores["mkt_logloss"])

print(f"Games: {len(test)}")
print(scores.round(4))
print(f"Share of the gap to the market closed since goals-only: {gap_closed:.0%}")

Games: 1921
prev_logloss     0.9778
model_logloss    0.9754
mkt_logloss      0.9592
dtype: float64
Share of the gap to the market closed since goals-only: 27%
